In [1]:
import pandas as pd

# =========================
# 1. 读取原始数据
# =========================

INPUT_FILE = "AI_industries_20260601.csv"
OUTPUT_FILE = "global_ai_industry_hotspots_top10.csv"

df = pd.read_csv(INPUT_FILE)

# =========================
# 2. 数据清洗
# =========================

df["industry"] = df["industry"].astype(str).str.strip()
df["reason"] = df["reason"].astype(str).str.strip()

df = df.dropna(subset=["industry"])
df = df[df["industry"] != ""]

# rank转数字
df["rank"] = pd.to_numeric(df["rank"], errors="coerce")

# 删除异常rank
df = df[(df["rank"] >= 1) & (df["rank"] <= 10)]

# =========================
# 3. 计算热度分数
# =========================
# rank1=10分
# rank2=9分
# ...
# rank10=1分

df["score"] = 11 - df["rank"]

# =========================
# 4. 汇总行业热度
# =========================

industry_summary = (
    df.groupby("industry")
      .agg(
          total_score=("score", "sum"),
          mentions=("industry", "count"),
          avg_rank=("rank", "mean")
      )
      .reset_index()
)

# =========================
# 5. 提取最佳理由
# =========================
# 选择该行业出现时排名最高（rank最小）的一条reason
# 如果多个模型都把它排第1，则自动取第一条

best_reason = (
    df.sort_values(
        by=["industry", "rank"]
    )
    .groupby("industry", as_index=False)
    .first()[["industry", "reason"]]
)

# =========================
# 6. 合并
# =========================

result = industry_summary.merge(
    best_reason,
    on="industry",
    how="left"
)

# =========================
# 7. 排序
# =========================
# 优先看总分
# 再看出现次数
# 再看平均排名

result = result.sort_values(
    by=[
        "total_score",
        "mentions",
        "avg_rank"
    ],
    ascending=[
        False,
        False,
        True
    ]
)

# =========================
# 8. 添加最终排名
# =========================

result = result.reset_index(drop=True)

result.insert(
    0,
    "rank",
    range(1, len(result) + 1)
)

# =========================
# 9. 输出Top10
# =========================

top10 = result.head(10)

top10.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# =========================
# 10. 显示结果
# =========================

print("\n===== GLOBAL AI INDUSTRY HOTSPOTS TOP10 =====\n")

print(
    top10[
        [
            "rank",
            "industry",
            "total_score",
            "mentions",
            "avg_rank",
            "reason"
        ]
    ]
)

print(f"\nSaved to: {OUTPUT_FILE}")


===== GLOBAL AI INDUSTRY HOTSPOTS TOP10 =====

   rank                         industry  total_score  mentions  avg_rank  \
0     1                        AI Agents           40         4  1.000000   
1     2                         AI Chips           33         4  2.750000   
2     3                AI Infrastructure           33         4  2.750000   
3     4           AI Enterprise Software           26         4  4.500000   
4     5  AI Research & Foundation Models           19         4  6.250000   
5     6                      AI Security           15         4  7.250000   
6     7               AI Cloud Platforms           13         3  6.666667   
7     8                Humanoid Robotics           11         4  8.250000   
8     9                    AI Healthcare           10         3  7.666667   
9    10           AI Data Infrastructure            9         2  6.500000   

                                              reason  
0  AI agents remain central as companies deploy a